# Feature - Development Classification

## Importing Relevant Libraries

In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import seaborn as sns
import kagglehub
import torch
import cv2
from scipy.spatial import distance
from torchvision.datasets import ImageFolder
from torchvision import transforms

## Loading and Converting Dataset

### Setting up Transformers

In [6]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
]) 

### Define folder paths

In [17]:
base_path = "../dataset/Dataset"
train_path = f"{base_path}/Images/Emotion/train"
test_path  = f"{base_path}/Images/Emotion/test"

### Creating the datasets

In [20]:
train_dataset = ImageFolder(train_path, transform=transform)
test_dataset = ImageFolder(test_path, transform=transform)

## Feature Extraction

### Feature Extraction Logic

Features to Extract from the images


*   **Fill ratio** - (Pixels with drawing) / (Total Pixels)

tiny figures in the corner (timidity/anxiety) or fill the whole page (confidence/impulsivity)

*   **Pressure/Stroke Thickness** - Average pixel intensity of the edges.

Heavy, dark lines often indicate high energy or aggression. Faint, sketchy lines can indicate hesitation or low energy.



*   **Color Palette (Warm vs. Cool)** - Dominant Hue analysis.

Predominance of Red/Orange (Warm) vs. Blue/Green (Cool) vs. Black/White.


*   **"Blob" Count (Object Individuation)** - Number of closed contours.

Younger children draw disconnected items. Older children connect them into a scene.


*   Centroid Logic (Centrality):

Is the main subject in the center (egocentric) or spread out?

In [22]:
class DrawingAnalyser:
    def __init__(self, image_path):
        self.image_path = image_path
        self.img_color = cv2.imread(self.image_path)
        
        if self.img_color is None:
            self.valid = False
            return 
        self.valid = True
        
        ## Resize for consistent processing
        h, w = self.img_color.shape[:2]
        scale = 500 / w
        dim = (500, int(h * scale))
        self.img_color = cv2.resize(self.img_color, dim, interpolation=cv2.INTER_AREA)
        self.img_gray = cv2.cvtColor(self.img_color, cv2.COLOR_BGR2GRAY)
        
        self.height, self.width = self.img_gray.shape
        self.total_pixels = self.height * self.width
        
    def _thresholding(self):
        """Thresholding"""
        _, thresh = cv2.threshold(self.img_gray, 200, 255, cv2.THRESH_BINARY_INV)
        return thresh
    
    def _space_utilization(self, thresh):
        """Space utilization"""
        non_zero = cv2.countNonZero(thresh)
        fill_ratio = non_zero / self.total_pixels
        return fill_ratio
    
    def _pressure(self, thresh, non_zero):
        """Average Pressure"""
        if non_zero == 0:
            return 0
        
        mask = thresh > 0
        raw_values = self.img_gray[mask]
        avg_pressure = np.mean(255 - raw_values)
        return avg_pressure
    
    def _color_palette(self):
        """Color palette"""
        hsv = cv2.cvtColor(self.img_color, cv2.COLOR_BGR2HSV)
        h, s, v = cv2.split(hsv)
        
        saturation_mask = s > 40
        valid_heus = h[saturation_mask]
        
        if len(valid_heus) == 0:
            return 0.5 # neutral / unknown 
        
        warm_count = 0
        cool_count = 0
        
        for hue in valid_heus:
            if (0 <= hue <= 35) or (170 <= hue <= 180):
                warm_count += 1
            elif (35 < hue < 170):
                cool_count += 1
                
        total = warm_count + cool_count + 1e-5
        warm_ratio = warm_count / total
        
        return warm_ratio
    
    def _blob_count(self, thresh):
        """Blob count"""
        contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        return len(contours)
    
    def _centrality(self, thresh, non_zero):
        if non_zero == 0:
            return 0
        
        M = cv2.moments(thresh)
        if M["m00"] == 0:
            return 0
        
        cX = int(M["m10"] / M["m00"])
        cY = int(M["m01"] / M["m00"])
        image_center = (self.width // 2, self.height // 2)
        
        dist_from_center = distance.euclidean((cX, cY), image_center)
        max_dist = distance.euclidean((0, 0), image_center)
        
        return 1.0 - (dist_from_center / max_dist)
    
    
    def analyse(self):
        """Public API method to analyse the dataset"""
        if not self.valid:
            return None
        thresh = self._thresholding()
        fill_ratio, non_zero = self._space_utilization(thresh)
        avg_pressure = self._pressure(thresh, non_zero)
        warm_ratio = self._color_palette()
        blob_count = self._blob_count(thresh)
        centrality = self._centrality(thresh, non_zero)
        
        return {
            "fill_ratio": round(fill_ratio, 4),
            "avg_pressure": round(avg_pressure, 2),
            "warm_color_ratio": round(warm_ratio, 2),
            "blob_count": blob_count,
            "centrality": round(centrality, 4),
        }   

### Function to analyse drawings using the class 

In [23]:
def analyse_drawing(image_path):
    analyser = DrawingAnalyser(image_path)
    return analyser.analyse()